In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

df = pd.read_parquet('../data/processed/protein_features_hol.parquet')
print("Loaded:", df.shape, "| LightGBM", lgb.__version__)

VALID_START, TEST_START = "2017-07-15", "2017-07-31"
train = df[df["date"] <  VALID_START]
valid = df[(df["date"] >= VALID_START) & (df["date"] < TEST_START)]
test  = df[df["date"] >= TEST_START]

print(f"Train {len(train):,} | Valid {len(valid):,} | Test {len(test):,}")

Loaded: (7459611, 45) | LightGBM 4.7.0
Train 7,205,999 | Valid 129,573 | Test 124,039


In [2]:
# Everything except identifiers and the target
DROP = ["date", "unit_sales"]

HOLIDAY_FEATS = [
    "hol_national", "hol_regional", "hol_local", "is_holiday",
    "days_to_holiday", "days_from_holiday",
    "is_earthquake", "is_event", "is_workday_makeup",
]

FEATURES_BASE = [c for c in df.columns if c not in DROP + HOLIDAY_FEATS]
FEATURES_HOL  = [c for c in df.columns if c not in DROP]

CATEGORICALS = ["family", "city", "state", "store_type", "store_nbr", "item_nbr", "class", "cluster"]

print(f"Base features: {len(FEATURES_BASE)}")
print(f"With holidays: {len(FEATURES_HOL)}")
print("\nBase:", FEATURES_BASE)

Base features: 34
With holidays: 43

Base: ['store_nbr', 'item_nbr', 'onpromotion', 'family', 'class', 'city', 'state', 'store_type', 'cluster', 'dayofweek', 'day', 'month', 'year', 'weekofyear', 'is_weekend', 'is_payday', 'days_in_month', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'roll_mean_7', 'roll_std_7', 'roll_max_7', 'roll_mean_14', 'roll_std_14', 'roll_max_14', 'roll_mean_28', 'roll_std_28', 'roll_max_28', 'promo_lag_1', 'promo_count_28', 'days_since_sale', 'zero_rate_28']


In [3]:
for c in CATEGORICALS:
    df[c] = df[c].astype("category")

train = df[df["date"] <  VALID_START]
valid = df[(df["date"] >= VALID_START) & (df["date"] < TEST_START)]

y_train, y_valid = train["unit_sales"], valid["unit_sales"]

# log1p transform — sales are heavily right-skewed
y_train_log = np.log1p(y_train)
y_valid_log = np.log1p(y_valid)

dtrain = lgb.Dataset(train[FEATURES_BASE], y_train_log, categorical_feature=CATEGORICALS, free_raw_data=False)
dvalid = lgb.Dataset(valid[FEATURES_BASE], y_valid_log, reference=dtrain, categorical_feature=CATEGORICALS, free_raw_data=False)

print("Datasets built.")

Datasets built.


In [ ]:
# Training 2 models now

In [4]:
def mae(y, p):  return np.mean(np.abs(y - p))
def rmse(y, p): return np.sqrt(np.mean((y - p) ** 2))
def wape(y, p): return np.sum(np.abs(y - p)) / np.sum(y) * 100
def rmsle(y, p):
    p = np.clip(p, 0, None)
    return np.sqrt(np.mean((np.log1p(p) - np.log1p(y)) ** 2))

def report(name, y, p, store):
    store.append({"model": name, "MAE": round(mae(y,p),4), "RMSE": round(rmse(y,p),4),
                  "WAPE": round(wape(y,p),2), "RMSLE": round(rmsle(y,p),4)})
    return store

In [5]:
# Train Base model

params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.1,
    "num_leaves": 63,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "num_threads": 4,
    "verbose": -1,
    "seed": 42,
}

model_base = lgb.train(
    params,
    dtrain,
    num_boost_round=400,
    valid_sets=[dvalid],
    valid_names=["valid"],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)

print("\nBest iteration:", model_base.best_iteration)

Training until validation scores don't improve for 30 rounds
[50]	valid's rmse: 0.596031
[100]	valid's rmse: 0.591774
[150]	valid's rmse: 0.589989
[200]	valid's rmse: 0.588776
[250]	valid's rmse: 0.588015
[300]	valid's rmse: 0.587505
[350]	valid's rmse: 0.587108
[400]	valid's rmse: 0.586786
Did not meet early stopping. Best iteration is:
[400]	valid's rmse: 0.586786

Best iteration: 400


In [6]:
results = []

# Predictions come back in log space — invert with expm1
pred_log = model_base.predict(valid[FEATURES_BASE], num_iteration=model_base.best_iteration)
pred_base = np.clip(np.expm1(pred_log), 0, None)
y_true = y_valid.values

results = report("LightGBM (no holidays)", y_true, pred_base, results)

# Baselines for reference
results = report("Moving avg 7d (baseline)", y_true, valid["roll_mean_7"].values, results)
results = report("Seasonal naive (baseline)", y_true, valid["lag_7"].values, results)

print(pd.DataFrame(results).to_string(index=False))

                    model    MAE    RMSE      WAPE  RMSLE
   LightGBM (no holidays) 3.2617  8.7001 41.020000 0.5867
 Moving avg 7d (baseline) 3.9725 13.6616 49.950001 0.7042
Seasonal naive (baseline) 4.2901 10.2457 53.950001 0.8342


In [7]:
# the holiday comparison, the question this phase exists to answer:

dtrain_h = lgb.Dataset(train[FEATURES_HOL], y_train_log, categorical_feature=CATEGORICALS, free_raw_data=False)
dvalid_h = lgb.Dataset(valid[FEATURES_HOL], y_valid_log, reference=dtrain_h, categorical_feature=CATEGORICALS, free_raw_data=False)

model_hol = lgb.train(
    params, dtrain_h, num_boost_round=400,
    valid_sets=[dvalid_h], valid_names=["valid"],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(100)],
)

pred_hol = np.clip(np.expm1(model_hol.predict(valid[FEATURES_HOL], num_iteration=model_hol.best_iteration)), 0, None)
results = report("LightGBM (with holidays)", y_true, pred_hol, results)

print("\n", pd.DataFrame(results).sort_values("WAPE").to_string(index=False))

Training until validation scores don't improve for 30 rounds
[100]	valid's rmse: 0.591997
[200]	valid's rmse: 0.588878
[300]	valid's rmse: 0.587345
[400]	valid's rmse: 0.586671
Did not meet early stopping. Best iteration is:
[400]	valid's rmse: 0.586671

                     model    MAE    RMSE      WAPE  RMSLE
 LightGBM (with holidays) 3.2528  8.2649 40.900000 0.5866
   LightGBM (no holidays) 3.2617  8.7001 41.020000 0.5867
 Moving avg 7d (baseline) 3.9725 13.6616 49.950001 0.7042
Seasonal naive (baseline) 4.2901 10.2457 53.950001 0.8342


In [9]:
# feature importance:

imp = pd.DataFrame({
    "feature": model_hol.feature_name(),
    "gain": model_hol.feature_importance("gain"),
}).sort_values("gain", ascending=False)
imp["gain_pct"] = (imp["gain"] / imp["gain"].sum() * 100).round(2)

print("Top 20 features:")
print(imp.head(20).to_string(index=False))

print("\nHoliday features:")
print(imp[imp["feature"].isin(HOLIDAY_FEATS)].to_string(index=False))

Top 20 features:
        feature         gain  gain_pct
   roll_mean_14 1.497755e+07     50.93
   roll_mean_28 6.502938e+06     22.11
    roll_mean_7 2.136442e+06      7.26
          lag_7 1.034200e+06      3.52
       item_nbr 6.059208e+05      2.06
         lag_14 5.197888e+05      1.77
          lag_1 4.931697e+05      1.68
         lag_28 4.822947e+05      1.64
    onpromotion 4.683595e+05      1.59
      dayofweek 4.541770e+05      1.54
days_since_sale 3.542030e+05      1.20
      store_nbr 2.305117e+05      0.78
   zero_rate_28 2.225751e+05      0.76
            day 2.165152e+05      0.74
     weekofyear 1.343041e+05      0.46
     roll_std_7 1.081930e+05      0.37
          class 7.279576e+04      0.25
    promo_lag_1 7.196468e+04      0.24
     is_weekend 5.763210e+04      0.20
          month 3.544234e+04      0.12

Holiday features:
          feature         gain  gain_pct
days_from_holiday 30652.264610      0.10
  days_to_holiday 24430.879766      0.08
     hol_national 1855

In [10]:
import os
os.makedirs('../models', exist_ok=True)

model_hol.save_model('../models/lgbm_holidays.txt')
model_base.save_model('../models/lgbm_base.txt')
print("Saved.")

[LightGBM] [Fatal] Model file ../models/lgbm_holidays.txt is not available for writes


LightGBMError: Model file ../models/lgbm_holidays.txt is not available for writes